# Deploy ADK Agent to Cloud Run

This notebook securely deploys our Economic Development Officer agent to Google Cloud Run, exposing it as a REST API for the front-end team.

**Prerequisites:**
1. Open the Secrets panel (🔑) on the left.
2. Add a secret named `GEMINI_API_KEY` and paste your Gemini API key.
3. Add a secret named `SERVICE_ACCOUNT_JSON` and paste the raw JSON string of your BigQuery service account key.

In [ ]:
# Install required ADK dependencies
!pip install google-adk mcp pandas

In [ ]:
# Clone the repository containing the agent code
# (Replace with your actual GitHub URL if it is different)
!git clone https://github.com/mastan30/ATL-13-MicroGrants.git

In [ ]:
import os
from google.colab import userdata
from google.adk.cli.utils import agent_loader
from google.adk.deploy import deployer

# 1. Securely fetch secrets from Colab's vault
secure_gemini_key = userdata.get('GEMINI_API_KEY')
service_account_json = userdata.get('SERVICE_ACCOUNT_JSON')

# 2. Define the path to the cloned agent folder
agent_path = "/content/ATL-13-MicroGrants/agent"

# 3. Load the ADK Agent from the source code
my_agent = agent_loader.load_agent(agent_path)

# 4. Define the deployment configuration
deployment_config = {
    "project": "qwiklabs-gcp-04-e4952d3717e9",
    "region": "us-central1",
    "service_name": "anchor-agent-api",
    
    # Inject the secure variables directly into the Cloud Run container environment
    "env": {
        "GEMINI_API_KEY": secure_gemini_key,
        "SERVICE_ACCOUNT_JSON": service_account_json
    }
}

# 5. Trigger the deployment to Cloud Run
print("Starting Secure Cloud Run Deployment...")
deployment_url = deployer.deploy_to_cloud_run(
    agent=my_agent, 
    source_dir=agent_path, 
    **deployment_config
)

print(f"\n\u2705 Agent successfully deployed! API URL: {deployment_url}")